<a href="https://colab.research.google.com/github/iplaycos1-crypto/MBA6636/blob/main/Copy_of_MBA6636_3769291_Sun_DEA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment1 with R code by Lei Sun
## Business Problem
### The professor faces a problem when he purchase a diamond for his girl-friend. As a common consumer, he finds it difficult to make choice and evaluate the value. Therefore he collected all diamond pricing data from three wholesalers. The objective is to identify a high-quality diamond at a reasonable price.

In [22]:
install.packages("tidyverse","readxl")

Warning message in install.packages("tidyverse", "readxl"):
“'lib = "readxl"' is not writable”


ERROR: Error in install.packages("tidyverse", "readxl"): unable to install packages


In [18]:
library(tidyverse)
library(readxl)

In [3]:
# GitHub raw file link
data_url <- "https://raw.githubusercontent.com/iplaycos1-crypto/MBA6636/main/Professor_Proposes_Data.xlsx"


In [4]:
# Download the Excel file from My GitHub account
download.file(
  url = data_url,
  destfile = "Professor_Proposes_Data.xlsx",
  mode = "wb"
)

In [5]:
# Read the downloaded file
diamonds_data <- readxl::read_excel(
  path = "Professor_Proposes_Data.xlsx",
  sheet = "Data"
)

In [6]:
head(diamonds_data)

Carat,Colour,Clarity,Cut,Certification,Polish,Symmetry,Price,Wholesaler
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
0.92,I,SI2,G,AGS,V,V,3000,1
0.92,I,SI2,V,AGS,G,G,3000,1
0.82,F,SI2,I,GIA,X,X,3004,1
0.81,G,SI1,I,GIA,X,V,3004,1
0.90,J,VS2,V,GIA,V,V,3006,1
0.87,F,SI2,I,AGS,G,V,3007,1


In [7]:
diamonds_original <- diamonds_data    # keep the original data

In [8]:
diamonds_clean <- diamonds_data       # data after clean

## ---1: Understanding the Data---
### Inspect data quality and clean the data

In [9]:
dim(diamonds_data)

[1] 440   9

In [10]:
# 1. Number of observations (rows)
nrow(diamonds_data)

# 2. Number of variables (columns)
ncol(diamonds_data)

# 3. Display variable names and data types
str(diamonds_data)

[1] 440

[1] 9

tibble [440 × 9] (S3: tbl_df/tbl/data.frame)
 $ Carat        : num [1:440] 0.92 0.92 0.82 0.81 0.9 0.87 0.8 0.84 0.8 0.8 ...
 $ Colour       : chr [1:440] "I" "I" "F" "G" ...
 $ Clarity      : chr [1:440] "SI2" "SI2" "SI2" "SI1" ...
 $ Cut          : chr [1:440] "G" "V" "I" "I" ...
 $ Certification: chr [1:440] "AGS" "AGS" "GIA" "GIA" ...
 $ Polish       : chr [1:440] "V" "G" "X" "X" ...
 $ Symmetry     : chr [1:440] "V" "G" "X" "V" ...
 $ Price        : num [1:440] 3000 3000 3004 3004 3006 ...
 $ Wholesaler   : num [1:440] 1 1 1 1 1 1 1 1 1 1 ...


In [11]:
colSums(is.na(diamonds_clean))    # check missing values

Carat        Colour       Clarity           Cut Certification 
            0             0             0             0             0 
       Polish      Symmetry         Price    Wholesaler 
            0             0             0             0

In [12]:
sum(duplicated(diamonds_clean))  #count fully duplicated rows

[1] 51

#### *Two diamonds may have identical recorded characteristics and prices but still be separate legitimate observations. A duplicated row is a reason to investigate, not automatic proof of a data-entry error*

In [13]:
diamonds_clean[duplicated(diamonds_clean), ]

Carat,Colour,Clarity,Cut,Certification,Polish,Symmetry,Price,Wholesaler
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
0.80,I,VS1,I,AGS,I,I,3031,1
1.01,J,SI2,X,EGL,V,V,3139,2
1.01,J,SI2,X,EGL,V,V,3139,2
1.01,J,SI2,I,EGL,V,X,3139,2
0.22,F,SI1,V,IGI,G,G,330,3
0.23,F,SI1,X,IGI,G,G,360,3
0.28,E,VVS2,X,GIA,V,X,658,3
0.30,H,SI2,X,GIA,V,V,493,3
0.30,H,SI2,X,GIA,V,V,493,3


In [21]:
categorical_vars <- c(
  "Colour", "Clarity", "Cut",
  "Certification", "Polish",
  "Symmetry", "Wholesaler"
)

lapply(diamonds_clean[categorical_vars], unique)      # check inconsistent category labels

$Colour
[1] "I" "F" "G" "J" "D" "H" "E" "L" "K"

$Clarity
[1] "SI2"  "SI1"  "VS2"  "VS1"  "SI3"  "I2"   "I1"   "VVS2" "VVS1"

$Cut
[1] "G" "V" "I" "F" "X"

$Certification
[1] "AGS" "GIA" "EGL" "DOW" "IGI"

$Polish
[1] "V" "G" "X" "I" "v" "F"

$Symmetry
[1] "V" "G" "X" "I" "F"

$Wholesaler
[1] 1 2 3

In [23]:
diamonds_clean <- diamonds_clean %>%
  mutate(
    across(
      all_of(categorical_vars),
      ~ trimws(as.character(.x))
    )
  )                                                   # check invalid numerical values

In [24]:
summary(diamonds_clean$Carat)
summary(diamonds_clean$Price)

# Inspect potentially invalid observations
diamonds_clean %>%
  filter(
    is.na(Carat) | is.na(Price) |
      Carat <= 0 | Price <= 0
  )                                                    # check invalid numerical value

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0900  0.3000  0.8100  0.6693  1.0100  1.5800 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    160     520    2169    1717    3012    3145 

Carat,Colour,Clarity,Cut,Certification,Polish,Symmetry,Price,Wholesaler
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>


In [25]:
data.frame(
  Stage = c("Original", "Cleaned"),
  Rows = c(
    nrow(diamonds_original),
    nrow(diamonds_clean)
  ),
  Columns = c(
    ncol(diamonds_original),
    ncol(diamonds_clean)
  )
)

Stage,Rows,Columns
<chr>,<int>,<int>
Original,440,9
Cleaned,440,9


In [26]:
library(dplyr)
library(tibble)

variable_summary <- tibble(
  Variable = names(diamonds_clean),
  Data_Type = sapply(
    diamonds_clean,
    function(x) class(x)[1]
  ),
  Number_of_Unique_Values = sapply(
    diamonds_clean,
    function(x) dplyr::n_distinct(x, na.rm = TRUE)
  ),
  Missing_Values = sapply(
    diamonds_clean,
    function(x) sum(is.na(x))
  )
)

print(variable_summary)

# A tibble: 9 × 4
  Variable      Data_Type Number_of_Unique_Values Missing_Values
  <chr>         <chr>                       <int>          <int>
1 Carat         numeric                        50              0
2 Colour        character                       9              0
3 Clarity       character                       9              0
4 Cut           character                       5              0
5 Certification character                       5              0
6 Polish        character                       6              0
7 Symmetry      character                       5              0
8 Price         numeric                       197              0
9 Wholesaler    character                       3              0


In [28]:
data_quality <- tibble(
  Check = c(
    "Total observations",
    "Total variables",
    "Missing cells",
    "Fully duplicated rows",
    "Invalid carat values",
    "Invalid price values"
  ),
  Result = c(
    nrow(diamonds_clean),
    ncol(diamonds_clean),
    sum(is.na(diamonds_clean)),
    sum(duplicated(diamonds_clean)),
    sum(!is.na(diamonds_clean$Carat) &
          diamonds_clean$Carat <= 0),
    sum(!is.na(diamonds_clean$Price) &
          diamonds_clean$Price <= 0)
  )
)

print(data_quality)

# A tibble: 6 × 2
  Check                 Result
  <chr>                  <int>
1 Total observations       440
2 Total variables            9
3 Missing cells              0
4 Fully duplicated rows     51
5 Invalid carat values       0
6 Invalid price values       0


In [33]:
install.packages("moments")
library(moments)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



### ***Descriptive Statistics with numeric variables***

In [37]:
# Numerical summary
numeric_summary <- diamonds_clean %>%
  summarise(
    across(
      c(Carat, Price),
      list(
        n = ~sum(!is.na(.)),
        mean = ~mean(., na.rm = TRUE),
        median = ~median(., na.rm = TRUE),
        sd = ~sd(., na.rm = TRUE),
        min = ~min(., na.rm = TRUE),
        q1 = ~quantile(., 0.25, na.rm = TRUE),
        q3 = ~quantile(., 0.75, na.rm = TRUE),
        max = ~max(., na.rm = TRUE),
        skewness = ~moments::skewness(., na.rm = TRUE),
        kurtosis = ~moments::kurtosis(., na.rm = TRUE)
      ),
      .names = "{.col}_{.fn}"
    )
  )

print(numeric_summary, width= Inf)




# A tibble: 1 × 20
  Carat_n Carat_mean Carat_median Carat_sd Carat_min Carat_q1 Carat_q3 Carat_max
    <int>      <dbl>        <dbl>    <dbl>     <dbl>    <dbl>    <dbl>     <dbl>
1     440      0.669         0.81    0.380      0.09      0.3     1.01      1.58
  Carat_skewness Carat_kurtosis Price_n Price_mean Price_median Price_sd
           <dbl>          <dbl>   <int>      <dbl>        <dbl>    <dbl>
1        -0.0499           1.34     440      1717.         2169    1176.
  Price_min Price_q1 Price_q3 Price_max Price_skewness Price_kurtosis
      <dbl>    <dbl>    <dbl>     <dbl>          <dbl>          <dbl>
1       160      520    3012.      3145        -0.0518           1.18


### ***Descriptive Statistics with categorical variables***

In [39]:

library(dplyr)

# Categorical variables in the diamond dataset
categorical_vars <- c(
  "Colour", "Clarity", "Cut",
  "Certification", "Polish",
  "Symmetry", "Wholesaler"
)

# Frequency and percentage for each categorical variable
for (v in categorical_vars) {

  cat("\nVariable:", v, "\n")

  frequency_table <- as.data.frame(
    table(diamonds_clean[[v]], useNA = "ifany")
  )

  names(frequency_table) <- c("Category", "Frequency")

  frequency_table$Percentage <- round(
    frequency_table$Frequency /
      sum(frequency_table$Frequency) * 100,
    2
  )

  print(frequency_table)
}



Variable: Colour 
  Category Frequency Percentage
1        D        20       4.55
2        E        54      12.27
3        F        58      13.18
4        G        43       9.77
5        H        71      16.14
6        I        79      17.95
7        J        72      16.36
8        K        31       7.05
9        L        12       2.73

Variable: Clarity 
  Category Frequency Percentage
1       I1        82      18.64
2       I2        28       6.36
3      SI1       116      26.36
4      SI2       110      25.00
5      SI3        26       5.91
6      VS1        30       6.82
7      VS2        41       9.32
8     VVS1         2       0.45
9     VVS2         5       1.14

Variable: Cut 
  Category Frequency Percentage
1        F        59      13.41
2        G        49      11.14
3        I        86      19.55
4        V        97      22.05
5        X       149      33.86

Variable: Certification 
  Category Frequency Percentage
1      AGS        12       2.73
2      DOW         1   